In [0]:
# ============================================================
# VoeBem Analytics AI
# Notebook: 03_Silver_VRA
# Objetivo: transformar os dados Bronze VRA em dados confiáveis
# Camada: Silver
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "voebem"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

TABELA_BRONZE_VRA = f"{CATALOG}.{SCHEMA_BRONZE}.vra"
TABELA_SILVER_VRA = f"{CATALOG}.{SCHEMA_SILVER}.vra"

# Cria o schema Silver caso ainda não exista
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")

print(f"Origem : {TABELA_BRONZE_VRA}")
print(f"Destino: {TABELA_SILVER_VRA}")
print("Schema Silver preparado.")

In [0]:
# ============================================================
# Célula 2 — Leitura e diagnóstico da Bronze
# ============================================================

df_bronze = spark.table(TABELA_BRONZE_VRA)

print(f"Registros Bronze: {df_bronze.count():,}")
print(f"Quantidade de colunas: {len(df_bronze.columns)}")

print("\nColunas disponíveis:")
for coluna in df_bronze.columns:
    print(f" - {coluna}")

df_bronze.printSchema()

In [0]:
# ============================================================
# Célula 3 — Padronização dos nomes das colunas
# ============================================================

df_silver = (
    df_bronze
    .withColumnRenamed("ICAO Empresa Aérea", "icao_empresa")
    .withColumnRenamed("Número Voo", "numero_voo")
    .withColumnRenamed("Código Autorização (DI)", "codigo_autorizacao")
    .withColumnRenamed("Código Tipo Linha", "codigo_tipo_linha")
    .withColumnRenamed("ICAO Aeródromo Origem", "icao_origem")
    .withColumnRenamed("ICAO Aeródromo Destino", "icao_destino")
    .withColumnRenamed("Partida Prevista", "partida_prevista")
    .withColumnRenamed("Partida Real", "partida_real")
    .withColumnRenamed("Chegada Prevista", "chegada_prevista")
    .withColumnRenamed("Chegada Real", "chegada_real")
    .withColumnRenamed("Situação Voo", "situacao_voo")
    .withColumnRenamed("Código Justificativa", "codigo_justificativa")
)

print(f"Registros: {df_silver.count():,}")
print(f"Colunas: {len(df_silver.columns)}")

print("\nColunas Silver:")
for coluna in df_silver.columns:
    print(f" - {coluna}")

In [0]:
# ============================================================
# CÉLULA 4 — Preparação da camada Silver
# Padronização dos nomes das colunas
# ============================================================

# Sempre reconstruímos a partir da Bronze
df_silver = spark.table(TABELA_BRONZE_VRA)

# Padronização dos nomes das colunas
df_silver = (
    df_silver
    .withColumnRenamed("ICAO Empresa Aérea", "icao_empresa")
    .withColumnRenamed("Número Voo", "numero_voo")
    .withColumnRenamed("Código Autorização (DI)", "codigo_autorizacao")
    .withColumnRenamed("Código Tipo Linha", "codigo_tipo_linha")
    .withColumnRenamed("ICAO Aeródromo Origem", "icao_origem")
    .withColumnRenamed("ICAO Aeródromo Destino", "icao_destino")
    .withColumnRenamed("Partida Prevista", "partida_prevista")
    .withColumnRenamed("Partida Real", "partida_real")
    .withColumnRenamed("Chegada Prevista", "chegada_prevista")
    .withColumnRenamed("Chegada Real", "chegada_real")
    .withColumnRenamed("Situação Voo", "situacao_voo")
    .withColumnRenamed("Código Justificativa", "codigo_justificativa")
)

# Validação simples
print(f"Registros Silver: {df_silver.count():,}")
print(f"Colunas Silver: {len(df_silver.columns)}")

print("\nColunas:")
for coluna in df_silver.columns:
    print(coluna)

# Amostra
display(df_silver.limit(10))

In [0]:
# ============================================================
# CÉLULA 5 — Qualidade e métricas da camada Silver
# ============================================================

from pyspark.sql import functions as F

# Conversão segura das datas
# Valores inválidos como "null" tornam-se NULL, sem interromper o pipeline

df_silver = (
    df_silver
    .withColumn(
        "partida_prevista_ts",
        F.expr("try_cast(partida_prevista AS TIMESTAMP)")
    )
    .withColumn(
        "partida_real_ts",
        F.expr("try_cast(partida_real AS TIMESTAMP)")
    )
    .withColumn(
        "chegada_prevista_ts",
        F.expr("try_cast(chegada_prevista AS TIMESTAMP)")
    )
    .withColumn(
        "chegada_real_ts",
        F.expr("try_cast(chegada_real AS TIMESTAMP)")
    )
)

# Calcula atraso da partida em minutos
df_silver = df_silver.withColumn(
    "atraso_partida_min",
    F.when(
        F.col("partida_prevista_ts").isNotNull()
        & F.col("partida_real_ts").isNotNull(),
        (
            F.unix_timestamp("partida_real_ts")
            - F.unix_timestamp("partida_prevista_ts")
        ) / 60
    )
)

# Calcula atraso da chegada em minutos
df_silver = df_silver.withColumn(
    "atraso_chegada_min",
    F.when(
        F.col("chegada_prevista_ts").isNotNull()
        & F.col("chegada_real_ts").isNotNull(),
        (
            F.unix_timestamp("chegada_real_ts")
            - F.unix_timestamp("chegada_prevista_ts")
        ) / 60
    )
)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print(f"Registros Silver: {df_silver.count():,}")
print(f"Colunas Silver: {len(df_silver.columns)}")

display(
    df_silver.select(
        "icao_empresa_aerea",
        "numero_voo",
        "icao_aerodromo_origem",
        "icao_aerodromo_destino",
        "partida_prevista_ts",
        "partida_real_ts",
        "atraso_partida_min",
        "chegada_prevista_ts",
        "chegada_real_ts",
        "atraso_chegada_min",
        "situacao_voo"
    ).limit(20)
)

In [0]:
# ============================================================
# CÉLULA 6 — Persistência da camada Silver em Delta
# ============================================================

TABELA_SILVER_VRA = f"{CATALOG}.{SCHEMA_SILVER}.vra"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_VRA)
)

print(f"Tabela Silver criada: {TABELA_SILVER_VRA}")

# Validação da tabela persistida
df_validacao = spark.table(TABELA_SILVER_VRA)

print(f"Registros persistidos: {df_validacao.count():,}")
print(f"Colunas persistidas: {len(df_validacao.columns)}")

display(
    df_validacao.select(
        "icao_empresa_aerea",
        "numero_voo",
        "icao_aerodromo_origem",
        "icao_aerodromo_destino",
        "partida_prevista_ts",
        "partida_real_ts",
        "atraso_partida_min",
        "chegada_prevista_ts",
        "chegada_real_ts",
        "atraso_chegada_min",
        "situacao_voo"
    ).limit(20)
)